# Wikipedia RAG Chatbot — Walkthrough

**Author:** Sabal Nemkul  
**Module:** COS6031-D Applied AI Professional — Element 3 portfolio  
**Project tier:** Semi-complex (Gen AI / retrieval)

## What this notebook does

Walks through the full RAG pipeline step by step:

1. Fetch a few Wikipedia articles
2. Chunk them into overlapping passages
3. Embed the chunks with a sentence-transformer model
4. Build a FAISS index for fast similarity search
5. Retrieve relevant chunks for a user question
6. Send chunks + question to Claude, get an answer

For a polished interactive UI use `streamlit run src/app.py` instead. This notebook is for understanding the internals.

## 1. Setup

Make sure you've installed the requirements and set your `ANTHROPIC_API_KEY` (e.g. via a `.env` file).

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path('..').resolve() / 'src'))

assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY before running.'

## 2. Ingest a small corpus

Three articles for speed in this walkthrough.

In [ ]:
from ingest import ingest, save_chunks, DATA_DIR

topics = ['Solar power', 'Wind power', 'Hydroelectricity']
chunks = ingest(topics, target_chars=800, overlap_chars=100)
print(f'Total chunks: {len(chunks)}')
print(f'Example: {chunks[0].chunk_id} ({chunks[0].char_count} chars)')
print(chunks[0].text[:300] + '...')

## 3. Build the FAISS index

In [ ]:
from dataclasses import asdict
from embed import build_index, save_index, INDEX_DIR

chunk_dicts = [asdict(c) for c in chunks]
index, embeddings = build_index(chunk_dicts)
save_index(index, embeddings, chunk_dicts, 'sentence-transformers/all-MiniLM-L6-v2')
print(f'\nIndex ready: {index.ntotal} vectors @ {embeddings.shape[1]} dims')

## 4. Retrieve top-k chunks for a sample query

In [ ]:
from rag import Retriever

retriever = Retriever()
query = 'How does a wind turbine generate electricity?'

results = retriever.retrieve(query, k=3)
for i, r in enumerate(results, 1):
    print(f'\n[{i}] {r.source}  (similarity = {r.score:.3f})')
    print(r.text[:300] + '...')

## 5. Generate an answer with Claude

The system prompt tells Claude to use only the supplied context and to cite the source articles.

In [ ]:
from rag import RAGChatbot

bot = RAGChatbot(k=4)
response = bot.ask('How does a wind turbine generate electricity?')

print(response.answer)
print()
print('Sources:')
for c in response.retrieved:
    print(f'  - {c.source} ({c.score:.3f})')
print(f'\nElapsed: {response.elapsed_seconds:.2f}s')

## 6. Out-of-scope question — refusal behaviour

The bot should refuse rather than hallucinate when asked something outside the corpus.

In [ ]:
response = bot.ask('Who won the 2024 Premier League title?')
print(response.answer)

## 7. Reflection

**What worked.**
- The end-to-end pipeline is small (~600 lines across 4 files) but covers all the moving parts of a real RAG system.
- L2-normalised embeddings + `IndexFlatIP` gives clean cosine retrieval semantics.
- The system prompt forces the model to cite sources and refuse out-of-scope questions, which is the right default behaviour for a knowledge-grounded bot.

**What does not work yet.**
- No multi-turn memory: every question is independent.
- No re-ranking: a cross-encoder pass on top-20 retrievals would meaningfully improve precision.
- No formal evaluation: I have not built a benchmark question/source set, so I'm relying on qualitative inspection.

**Connection to my other work.**
- The chunking/embedding choices in this project are conceptually the same kind of *dataset construction* decisions that shaped the bias issue in my Industrial AI Project (Element 2 individual report). What the system can retrieve is determined entirely by what's in the index — exactly as what my classifier could detect was determined by what was in the training data.